# Gemini 3 Flash Preview 모델 테스트

이 노트북은 Google GenAI SDK를 사용하여 Gemini 3 Flash Preview 모델을 테스트합니다.

## 테스트 항목
1. **텍스트 생성**: 기본 텍스트 생성 기능
2. **이미지 분석**: Vision 기능 테스트
3. **멀티모달**: 텍스트 + 이미지 동시 처리
4. **스트리밍**: 실시간 응답 스트리밍
5. **시스템 인스트럭션**: 시스템 프롬프트 적용
6. **JSON 형식 응답**: 구조화된 응답 테스트
7. **성능 벤치마크**: 응답 시간 및 처리 속도 측정
8. **에러 핸들링**: 다양한 에러 상황 테스트


## 0. 필수 패키지 설치

먼저 필요한 패키지를 설치합니다. (이미 설치되어 있다면 스킵 가능)


In [4]:
# 필수 패키지 설치
import subprocess
import sys

packages_to_check = {
    "google-genai": "google.genai",
    "python-dotenv": "dotenv",
    "pillow": "PIL",
}

print("📦 필수 패키지 확인 중...\n")

for package_name, import_name in packages_to_check.items():
    try:
        # 패키지가 이미 설치되어 있는지 확인
        __import__(import_name)
        print(f"✅ {package_name} - 이미 설치됨")
    except ImportError:
        # 설치되지 않았으면 설치
        print(f"📥 {package_name} 설치 중...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name], 
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✅ {package_name} - 설치 완료")
        except subprocess.CalledProcessError:
            print(f"❌ {package_name} - 설치 실패 (수동 설치 필요: pip install {package_name})")

print("\n✅ 패키지 확인 완료!")


📦 필수 패키지 확인 중...

✅ google-genai - 이미 설치됨
✅ python-dotenv - 이미 설치됨
✅ pillow - 이미 설치됨

✅ 패키지 확인 완료!


## 1. 환경 설정 및 라이브러리 임포트

In [5]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
import io

# .env 파일 로드
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

# API 키 확인
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY가 .env 파일에 설정되지 않았습니다.")

# Google GenAI Client 초기화 (최신 SDK 방식)
client = genai.Client(api_key=API_KEY)

# 모델 설정
MODEL_NAME = "gemini-3-flash-preview"

print(f"✅ Google GenAI Client 초기화 완료")
print(f"📦 모델: {MODEL_NAME}")
print(f"🔑 API 키: {'설정됨' if API_KEY else '미설정'}")
print(f"📚 SDK: google-genai (최신 Client 방식)")


✅ Google GenAI Client 초기화 완료
📦 모델: gemini-3-flash-preview
🔑 API 키: 설정됨
📚 SDK: google-genai (최신 Client 방식)


## 2. 텍스트 생성 테스트


In [6]:
# 기본 텍스트 생성
prompt = "전기화재 감식에서 접촉불량을 판별하는 주요 지표 3가지를 설명해주세요."

print(f"📝 프롬프트: {prompt}")
print("\n" + "="*60)
print("🤖 모델 응답:")
print("="*60 + "\n")

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
        max_output_tokens=1000,
    )
)

print(response.text)


📝 프롬프트: 전기화재 감식에서 접촉불량을 판별하는 주요 지표 3가지를 설명해주세요.

🤖 모델 응답:

전기화재 감식에서 **접촉불량(Loose Connection)**은 전선과 단자, 혹은 전선과 전선 사이의 연결이 느슨해져 저항이 증가하고, 이로 인해 국부적인 발열이 발생하는 현상을 말합니다.

이를 판별하는 주요 지표 3가지는 다음과 같습니다.

---

### 1. 아산화동($Cu_2O$)의 증식 (Cuprous Oxide Formation)
접촉불량 판별에서 가장 결정적인 지표 중 하나입니다.

*   **원리:** 접촉부의 저항 증가로 인해 국부적인 고온 상태가 지속되면, 구리(도체)와 공기 중의 산소가 반응하여 **아산화동($Cu_2O$)**이라는 화합물이 생성됩니다.
*   **특징:** 아산화동은 반도체 성질을 가지고 있어 온도 상승에 따라 저항이 감소하면서 더 많은 전류를 흐르게 하고, 이는 다시 열을 발생시키는 '열 폭주' 현상을 일으킵니다. 
*   **감식 포인트:** 단자대나 전선 연결 부위에서 **적갈색 또는 루비색**의 결정체나 증식된 흔적이 발견되면 접촉불량에 의한 과열로 판단할 수 있습니다.

### 2. 다공성 용융흔 (Porous Arc Beads)
접촉불량은 한 번에 큰 폭발이 일어나는 단락(Short Circuit)과 달리, 미세한 아크(Scintillation)가 반복적으로 발생합니다.

*   **원리:** 접촉이 떨어졌다 붙었다를 반복하거나 접촉 면적이 좁아지면서 미세한 불꽃(아크)이 발생하고, 이


## 3. 이미지 분석 테스트


In [ ]:
# 테스트 이미지 로드
image_path = Path.cwd().parent / "data" / "Primary_Arc_Bead_1.png"

if not image_path.exists():
    print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
else:
    print(f"📷 이미지 로드: {image_path.name}")
    
    # 이미지 바이트 데이터 읽기
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    # 이미지 MIME 타입 감지
    if image_data[:4] == b'\x89PNG':
        mime_type = "image/png"
    elif image_data[:3] == b'\xff\xd8\xff':
        mime_type = "image/jpeg"
    else:
        mime_type = "image/png"
    
    print(f"📋 MIME 타입: {mime_type}")
    print(f"📏 이미지 크기: {len(image_data):,} bytes")
    
    # 이미지 분석 프롬프트
    vision_prompt = """이 이미지를 분석하여 다음 항목을 확인하세요:
1. 전선의 용융흔 위치
2. 탄화 정도
3. 색상 변화 (특히 붉은색/주황색 영역)
4. 전선의 구조적 손상

각 항목에 대해 관찰한 내용을 상세히 설명해주세요."""
    
    print("\n" + "="*60)
    print("🔍 이미지 분석 시작...")
    print("="*60 + "\n")
    
    # Vision API 호출 (최신 SDK 방식)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            vision_prompt,
            types.Part.from_bytes(data=image_data, mime_type=mime_type)
        ],
        config=types.GenerateContentConfig(
            temperature=0.7,
            max_output_tokens=2000,
        )
    )
    
    print(response.text)


## 4. 멀티모달 테스트 (텍스트 + 이미지)


In [ ]:
# 여러 이미지와 텍스트를 함께 처리
image_paths = [
    Path.cwd().parent / "data" / "Primary_Arc_Bead_1.png",
    Path.cwd().parent / "data" / "Primary_Arc_Bead_2.png",
]

multimodal_parts = []

# 첫 번째 텍스트 프롬프트
multimodal_parts.append("다음 두 이미지를 비교 분석해주세요.")

# 이미지들 추가
for i, img_path in enumerate(image_paths, 1):
    if img_path.exists():
        with open(img_path, "rb") as f:
            image_data = f.read()
        
        # MIME 타입 감지
        if image_data[:4] == b'\x89PNG':
            mime_type = "image/png"
        else:
            mime_type = "image/jpeg"
        
        multimodal_parts.append(
            types.Part.from_bytes(data=image_data, mime_type=mime_type)
        )
        print(f"✅ 이미지 {i} 로드: {img_path.name}")
    else:
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {img_path}")

# 추가 텍스트 프롬프트
multimodal_parts.append("\n두 이미지의 차이점과 공통점을 분석하고, 화재 원인 추론에 어떤 도움이 되는지 설명해주세요.")

if len(multimodal_parts) > 1:
    print("\n" + "="*60)
    print("🖼️ 멀티모달 분석 시작...")
    print("="*60 + "\n")
    
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=multimodal_parts,
        config=types.GenerateContentConfig(
            temperature=0.7,
            max_output_tokens=2000,
        )
    )
    
    print(response.text)
else:
    print("❌ 분석할 이미지가 없습니다.")


## 5. 스트리밍 응답 테스트


In [ ]:
# 스트리밍 응답
stream_prompt = "전기화재 감식의 전체 프로세스를 단계별로 설명해주세요."

print(f"📝 프롬프트: {stream_prompt}")
print("\n" + "="*60)
print("🌊 스트리밍 응답:")
print("="*60 + "\n")

# 스트리밍 응답 (최신 SDK 방식)
full_text = ""
for chunk in client.models.generate_content_stream(
    model=MODEL_NAME,
    contents=stream_prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
    )
):
    if chunk.text:
        print(chunk.text, end="", flush=True)
        full_text += chunk.text

print("\n\n" + "="*60)
print(f"✅ 총 {len(full_text)}자 수신 완료")


## 6. 시스템 인스트럭션 테스트


In [ ]:
# 시스템 인스트럭션을 포함한 콘텐츠 생성
system_instruction = """당신은 전기화재 감식 전문가입니다.
다음 원칙을 따라야 합니다:
1. 객관적 관찰과 해석을 분리하세요
2. 확실하지 않은 정보는 추측하지 마세요
3. 시각적 증거가 부족하면 "시각적 정보 부족으로 판단 불가"라고 명시하세요
4. 전문 용어를 정확하게 사용하되, 필요시 정의를 덧붙이세요"""

test_prompt = "이미지에서 아산화동(Cu₂O)을 확정적으로 탐지할 수 있나요?"

print(f"📝 프롬프트: {test_prompt}")
print("\n" + "="*60)
print("🎯 시스템 인스트럭션 적용 응답:")
print("="*60 + "\n")

# 시스템 인스트럭션을 config에 포함 (최신 SDK 방식)
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=test_prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
        system_instruction=system_instruction,
    )
)

print(response.text)


## 7. JSON 형식 응답 테스트


In [ ]:
# JSON 형식으로 응답 요청
json_prompt = """다음 이미지를 분석하여 JSON 형식으로 응답하세요:
{
    "location_type": "접속점 위치 (terminal/blade/screw/splicing/mid_span/unknown)",
    "carbonization_detected": true/false,
    "color_analysis": {
        "red_tone_present": true/false,
        "orange_tone_present": true/false
    },
    "confidence": 0-100,
    "reasoning": "판단 근거"
}"""

# 이미지 로드
image_path = Path.cwd().parent / "data" / "Primary_Arc_Bead_1.png"

if image_path.exists():
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    mime_type = "image/png" if image_data[:4] == b'\x89PNG' else "image/jpeg"
    
    print("\n" + "="*60)
    print("📋 JSON 형식 응답 테스트")
    print("="*60 + "\n")
    
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            json_prompt,
            types.Part.from_bytes(data=image_data, mime_type=mime_type)
        ],
        config=types.GenerateContentConfig(
            temperature=0.3,  # JSON 응답은 낮은 temperature 권장
            max_output_tokens=1000,
        )
    )
    
    print(response.text)
    
    # JSON 파싱 시도
    try:
        json_start = response.text.find('{')
        json_end = response.text.rfind('}') + 1
        if json_start != -1 and json_end > json_start:
            json_text = response.text[json_start:json_end]
            parsed_json = json.loads(json_text)
            print("\n" + "="*60)
            print("✅ JSON 파싱 성공:")
            print("="*60)
            print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
    except json.JSONDecodeError as e:
        print(f"\n⚠️ JSON 파싱 실패: {e}")
else:
    print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")


## 8. 성능 벤치마크


In [ ]:
import time

# 응답 시간 측정
test_prompts = [
    "전기화재 감식의 기본 원칙을 설명하세요.",
    "접촉불량의 주요 증거는 무엇인가요?",
    "단락흔과 용융흔의 차이를 설명하세요.",
]

print("⏱️ 성능 벤치마크 시작...\n")

results = []
for i, prompt in enumerate(test_prompts, 1):
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.7,
            max_output_tokens=500,
        )
    )
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    response_length = len(response.text) if hasattr(response, 'text') else 0
    
    results.append({
        "prompt": prompt[:50] + "...",
        "elapsed_time": elapsed_time,
        "response_length": response_length,
        "tokens_per_second": response_length / elapsed_time if elapsed_time > 0 else 0
    })
    
    print(f"테스트 {i}: {elapsed_time:.2f}초 ({response_length}자)")

print("\n" + "="*60)
print("📊 벤치마크 결과:")
print("="*60)

avg_time = sum(r["elapsed_time"] for r in results) / len(results)
avg_length = sum(r["response_length"] for r in results) / len(results)
avg_tps = sum(r["tokens_per_second"] for r in results) / len(results)

print(f"평균 응답 시간: {avg_time:.2f}초")
print(f"평균 응답 길이: {avg_length:.0f}자")
print(f"평균 처리 속도: {avg_tps:.1f}자/초")

print("\n상세 결과:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['prompt']}")
    print(f"     시간: {r['elapsed_time']:.2f}초, 길이: {r['response_length']}자")


## 9. 에러 핸들링 테스트


In [ ]:
# 다양한 에러 상황 테스트

# 1. 잘못된 이미지 데이터
print("테스트 1: 잘못된 이미지 데이터")
try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            "이 이미지를 분석하세요",
            types.Part.from_bytes(data=b"invalid_data", mime_type="image/png")
        ]
    )
    print("✅ 처리됨 (에러 없음)")
except Exception as e:
    print(f"❌ 에러 발생: {type(e).__name__}: {e}")

print("\n" + "-"*60 + "\n")

# 2. 매우 긴 프롬프트
print("테스트 2: 매우 긴 프롬프트")
long_prompt = "설명해주세요. " * 1000
try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=long_prompt
    )
    print(f"✅ 처리됨 (응답 길이: {len(response.text)}자)")
except Exception as e:
    print(f"❌ 에러 발생: {type(e).__name__}: {e}")

print("\n" + "-"*60 + "\n")

# 3. 빈 프롬프트
print("테스트 3: 빈 프롬프트")
try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=""
    )
    print(f"✅ 처리됨 (응답: {response.text[:100]}...)")
except Exception as e:
    print(f"❌ 에러 발생: {type(e).__name__}: {e}")


## 10. 요약 및 결론


In [ ]:
print("="*60)
print("📋 Gemini 3 Flash Preview 모델 테스트 요약")
print("="*60)
print(f"\n✅ 모델: {MODEL_NAME}")
print(f"✅ SDK: google-genai (최신 Client 방식)")
print(f"✅ 인증: API Key 기반")
print(f"✅ Import: from google import genai")
print("\n테스트 완료 항목:")
print("  ✓ 텍스트 생성")
print("  ✓ 이미지 분석")
print("  ✓ 멀티모달 처리")
print("  ✓ 스트리밍 응답")
print("  ✓ 시스템 인스트럭션")
print("  ✓ JSON 형식 응답")
print("  ✓ 성능 벤치마크")
print("  ✓ 에러 핸들링")
print("\n" + "="*60)
print("모든 테스트가 완료되었습니다! 🎉")
